<a href="https://colab.research.google.com/github/edugovea/Proyectos-Personales-Auditoria-Analisis-de-datos-y-Machine-Learning/blob/main/04-clasificador-sentimiento-nlp/notebooks/01_eda_modelo_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - EDA y Modelo Base | Clasificador de Sentimiento (Olist)

Continuación del Proyecto 2. Objetivo: clasificar reseñas de clientes en **positivas (4-5★)** y **negativas (1-2★)**, descartando las neutrales (3★).

**Decisiones de diseño:**
- Etiquetado **binario**, se descartan las 3★.
- Solo se usan reseñas **con texto** (`review_comment_message` no nulo).
- Idioma: portugués brasileño.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 120)

# Ruta local (repo clonado). En Colab no existe, así que se descarga de Kaggle.
RAW = '../data/raw/olist_order_reviews_dataset.csv'

if not os.path.exists(RAW):
    import kagglehub
    ruta = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
    RAW = os.path.join(ruta, 'olist_order_reviews_dataset.csv')

print('Dataset:', RAW)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset: /kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv


## 1. Carga y vista general

In [2]:
df = pd.read_csv(RAW)
print('Filas totales:', len(df))
df.head()

Filas totales: 99224


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


## 2. Calidad de datos: ¿cuántas reseñas tienen texto?

Hallazgo clave de auditoría: la mayoría de las reseñas solo traen estrellas, sin comentario. Para un clasificador de texto solo sirven las filas con comentario.

In [3]:
print('Con comentario:', df['review_comment_message'].notna().sum())
print('Sin comentario:', df['review_comment_message'].isna().sum())
print('\nDistribución de estrellas (todas):')
print(df['review_score'].value_counts().sort_index())

Con comentario: 40977
Sin comentario: 58247

Distribución de estrellas (todas):
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


## 3. Etiquetado: binario descartando 3★

In [4]:
# Solo filas con texto
txt = df[df['review_comment_message'].notna()].copy()

# Descartar neutrales (3★)
txt = txt[txt['review_score'] != 3].copy()

# Etiqueta binaria: 1 = positivo (4-5), 0 = negativo (1-2)
txt['sentimiento'] = (txt['review_score'] >= 4).astype(int)

print('Total utilizable:', len(txt))
print(txt['sentimiento'].value_counts(normalize=True).rename({0: 'negativo', 1: 'positivo'}))

Total utilizable: 37420
sentimiento
positivo    0.708979
negativo    0.291021
Name: proportion, dtype: float64


## 4. Exploración del texto
_(longitud de comentarios, palabras frecuentes por clase, etc.)_

In [5]:
txt['n_chars'] = txt['review_comment_message'].str.len()
txt.groupby('sentimiento')['n_chars'].describe()

,count,mean,std,min,25%,50%,75%,max
sentimiento,,,,,,,,
0,10890.0,99.332048,59.709641,1.0,48.0,89.0,150.0,208.0
1,26530.0,54.068187,43.934555,1.0,22.0,43.0,72.0,207.0


## 5. Baseline tonto (clase mayoritaria)

Vara a superar: un modelo que siempre predice 'positivo'. Si el modelo real no la supera con claridad, no aprendió nada útil.

In [6]:
baseline_acc = txt['sentimiento'].mean()  # proporción de positivos
print(f'Accuracy del baseline (siempre positivo): {baseline_acc:.3f}')

Accuracy del baseline (siempre positivo): 0.709


## 6. Modelo base: TF-IDF + Regresión Logística

Se usa `Pipeline` para evitar *data leakage*: el vectorizador se ajusta SOLO con el conjunto de entrenamiento.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

X = txt['review_comment_message']
y = txt['sentimiento']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=5, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['negativo', 'positivo']))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

    negativo       0.83      0.94      0.88      2178
    positivo       0.98      0.92      0.95      5306

    accuracy                           0.93      7484
   macro avg       0.90      0.93      0.91      7484
weighted avg       0.93      0.93      0.93      7484

[[2058  120]
 [ 436 4870]]


## 7. Análisis de errores _(sección estrella)_

Revisar reseñas mal clasificadas y agruparlas por patrón: sarcasmo, reseñas mixtas, portugués coloquial, etc. Documentar *dónde y por qué* falla el modelo lineal.

In [8]:
errores = X_test.to_frame()
errores['real'] = y_test.values
errores['pred'] = y_pred
errores = errores[errores['real'] != errores['pred']]
print('Total mal clasificadas:', len(errores))
errores.head(20)

Total mal clasificadas: 556


,review_comment_message,real,pred
46995,"Recebi o chaveiro na cor bronze, apesar de ter pedido o da cor preta, nas imagens do site, a cor era bem diferente d...",1,0
66878,"Recebi a máscara com trincas no acrílico frontal, como faço para reclamar?",1,0
89518,A loja fez a entrega antes do prazo mas o produto veio com bastante defeito de 104 sombras da paleta veio faltando 1...,0,1
49808,Chegou no prazo estipulado. Porém veio faltando umas peças pequenas que dá o Acabamento. A cadeira o material perceb...,0,1
48608,Só não gostei pq tive que ir buscar no correio,1,0
93562,"Somente tive problema na entrega. Quando o produto chegava em minha residência, eu não estava em casa. Houve desenco...",1,0
68183,Olá BOM DIA Ñ recebir meus pididos escova progressiva Salvatore Ok,0,1
90698,Demorou um pouco pra fazer a nota fiscal mas não tem problema.,1,0
70775,A foto da placa com a imagem ao redor dá para entender que era grande,1,0
41030,Os tampos da mesa estavam danificados e aparentemente tentaram pintar com qualquer tinta antes de enviar o que só pi...,0,1


In [9]:
# 7.b - Muestreo para categorización manual de errores
falsos_negativos = errores[errores['real'] == 1]  # positivas marcadas negativas
falsos_positivos = errores[errores['real'] == 0]  # negativas marcadas positivas

print(f'Falsas alarmas (real + / pred -): {len(falsos_negativos)} ({len(falsos_negativos)/len(errores):.0%})')
print(f'Quejas no detectadas (real - / pred +): {len(falsos_positivos)} ({len(falsos_positivos)/len(errores):.0%})')

# Muestra aleatoria reproducible para etiquetar a mano
muestra = errores.sample(30, random_state=42)
for i, fila in muestra.iterrows():
    print(f"\n[{i}] real={fila['real']} pred={fila['pred']}")
    print(fila['review_comment_message'][:300])

Falsas alarmas (real + / pred -): 436 (78%)
Quejas no detectadas (real - / pred +): 120 (22%)

[89857] real=1 pred=0
Produto de boa qualidade e acabamento para o preço que está sendo vendido. Apesar de não ser grande, cabe um notebook de até 17 polegadas confortavelmente. Tem zipers grandes com trilhos robustos. 

[24771] real=1 pred=0
Tem um bom aumento, mas ainda não é o que procuro pois, vc tem que ficar muito próximo, tem que quase encostar no objeto que observa.

[64107] real=1 pred=0
NAO TENHO NENHUMA RECLAMAÇAO A FAZER , MAS ATE ESTA DATA DE HOJE SO RECEBI A MOCHILA AINDA FALTA O SACO DE DORMIR ,ESTOU NO AGUARDO ,DESDE JA AGRADEÇO .

[84214] real=0 pred=1
Foi entregue na data,só acho que a taxa de entrega poderia ser mais em conta.


[652] real=0 pred=1
Dentro do prazo porém poderiam agilizar como no meu outro produto que foi entregue em 2 dias.

[84335] real=1 pred=0
Talvez não seja uma mochila de grande durabilidade mas pelo custo benéfico é um otimo produto 

[90798] real=1 p